In [1]:
import sys
sys.path.append('/Users/mariana/Documents/projects/Huawei/survan')
# sys.path.append('/home/mvargas/code/Huawei/survan')

from deep_lambda_cox import DeepLambdaSA
from lambda_cox import LambdaSA
from baseline_cox import SA

from utils import concordance_index, unroll_time, get_single_task_dataset, kaplan_meier, pad_to, get_single_target_and_mask
import yaml
import jax
import jax.numpy as jnp
import numpy as np
import haiku as hk
import matplotlib.pyplot as plt

In [2]:
config_path = '../configs/config_tasks.yaml'
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

In [3]:
model = SA(config, seed=42)

In [4]:
seqs = model.data['seqs']

In [5]:
X = seqs[:32]

In [6]:
X.shape

(32, 100, 39)

In [134]:
class TSTransformer(hk.Module):
    def __init__(self, hidden_size, seq_len, output_dim, num_layers=3, seed=42):
        super(TSTransformer, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.seq_len = seq_len
        self.output_dim = output_dim
        self.keys_seq = hk.PRNGSequence(seed)

    def __call__(self, x):
        x = hk.Linear(self.hidden_size)(x)
        mask = self._get_mask_future()
        w_init = hk.initializers.VarianceScaling(2 / self.num_layers)
        for _ in range(self.num_layers):
            x = hk.MultiHeadAttention(num_heads=4, 
                                      key_size=16, 
                                      model_size=self.hidden_size,
                                      w_init=w_init)(x, x, mask)
            x = hk.LayerNorm(axis=-1, create_scale=True, create_offset=True)(x)
            x = hk.Linear(self.hidden_size)(x)
            x = hk.dropout(next(self.keys_seq), 0.0, x)

        return x

    def _get_mask_future(self):
        n = self.seq_len
        mask = jnp.tril(jnp.ones((n, n), dtype=jnp.float32))
        return mask

# Example usage:

In [147]:
horizon = 100

def forward_fn(inputs):
    ts_transformer = TSTransformer(hidden_size=64, 
                                   seq_len=100,
                                   output_dim=16,
                                   num_layers=3)
    x = ts_transformer(inputs)
    x = hk.Linear(horizon)(x)
    return x 


rng = jax.random.PRNGKey(0)
_some_input = X[:2]
forward = hk.without_apply_rng(hk.transform(forward_fn))
params = forward.init(rng, _some_input)
forward = forward.apply

In [148]:
out = forward(params, X)

In [149]:
out.shape

(32, 100, 100)

In [150]:
X.shape

(32, 100, 39)

In [38]:
def _get_mask_future(n):
    mask = jnp.triu(jnp.ones((n, n), dtype=jnp.float32), 1)
    mask = jax.lax.cond(jnp.size(mask) > 0, lambda mask: jnp.where(mask == 1, jnp.float32('-inf'), mask), lambda x: x, operand=mask)
    return mask

In [119]:
mask = _get_mask_future(32)
mask

Array([[  0., -inf, -inf, ..., -inf, -inf, -inf],
       [  0.,   0., -inf, ..., -inf, -inf, -inf],
       [  0.,   0.,   0., ..., -inf, -inf, -inf],
       ...,
       [  0.,   0.,   0., ...,   0., -inf, -inf],
       [  0.,   0.,   0., ...,   0.,   0., -inf],
       [  0.,   0.,   0., ...,   0.,   0.,   0.]], dtype=float32)

In [118]:
jnp.tril(jnp.ones((32,32)))

Array([[1., 0., 0., ..., 0., 0., 0.],
       [1., 1., 0., ..., 0., 0., 0.],
       [1., 1., 1., ..., 0., 0., 0.],
       ...,
       [1., 1., 1., ..., 1., 0., 0.],
       [1., 1., 1., ..., 1., 1., 0.],
       [1., 1., 1., ..., 1., 1., 1.]], dtype=float32)

In [113]:
import haiku as hk
import jax
import jax.numpy as jnp

def my_attention_module(model_size, key_size):
    return MyAttentionModule(model_size=model_size, key_size=key_size)

# Define a simple Haiku module with MultiHeadAttention
class MyAttentionModule(hk.Module):
    def __init__(self, model_size, key_size, num_layers=1):
        super(MyAttentionModule, self).__init__()
        self.model_size = model_size
        self.key_size = key_size
        self.num_layers = num_layers

    def __call__(self, x):
        # Apply MultiHeadAttention
        initializer = hk.initializers.VarianceScaling(2 / 100)
        mask = _get_mask_future(x.shape[1])
        mask = jnp.ones((x.shape[1], x.shape[1]))
        attn_output = hk.MultiHeadAttention(num_heads=1, 
                                            model_size=self.model_size, 
                                            key_size=2,
                                            w_init_scale=0.01)(x, x, mask)
        return attn_output

# Example usage within hk.transform
def forward_fn(x):
    attention_module = my_attention_module(model_size=4, key_size=16)
    return attention_module(x)

# Initialize the transformed function
transformed_fn = hk.transform(forward_fn)

# Example input
input_embeddings = jnp.ones((32, 10, 64), dtype=jnp.float32)

# Apply the transformed function to the input embeddings
rng = jax.random.PRNGKey(0)
params = transformed_fn.init(rng, input_embeddings)

# Apply the attention module to the input embeddings
output = transformed_fn.apply(params, rng, input_embeddings)

print("Input shape:", input_embeddings.shape)
print("Output shape:", output.shape)


Input shape: (32, 10, 64)
Output shape: (32, 10, 4)


In [114]:
output.shape

(32, 10, 4)

In [115]:
output

Array([[[ 0.0017094 , -0.01472762,  0.00302672, -0.00317471],
        [ 0.0017094 , -0.01472762,  0.00302672, -0.00317471],
        [ 0.0017094 , -0.01472762,  0.00302672, -0.00317471],
        ...,
        [ 0.0017094 , -0.01472762,  0.00302672, -0.00317471],
        [ 0.0017094 , -0.01472762,  0.00302672, -0.00317471],
        [ 0.0017094 , -0.01472762,  0.00302672, -0.00317471]],

       [[ 0.0017094 , -0.01472762,  0.00302672, -0.00317471],
        [ 0.0017094 , -0.01472762,  0.00302672, -0.00317471],
        [ 0.0017094 , -0.01472762,  0.00302672, -0.00317471],
        ...,
        [ 0.0017094 , -0.01472762,  0.00302672, -0.00317471],
        [ 0.0017094 , -0.01472762,  0.00302672, -0.00317471],
        [ 0.0017094 , -0.01472762,  0.00302672, -0.00317471]],

       [[ 0.0017094 , -0.01472762,  0.00302672, -0.00317471],
        [ 0.0017094 , -0.01472762,  0.00302672, -0.00317471],
        [ 0.0017094 , -0.01472762,  0.00302672, -0.00317471],
        ...,
        [ 0.0017094 , -0.01